In [1]:
import os
import time
import urllib.request
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import pandas as pd
import numpy as np
from pytesseract import Output
import pytesseract
import cv2
import io
#import matplotlib.pyplot as plt

from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service 
from bs4 import BeautifulSoup

import urllib.request

from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException

from PIL import Image, ImageOps, ImageFilter
from skimage import io
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects

## 이미지 크롤링 후 OCR 진행

In [20]:
def initialize_driver():
    chrome_options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=Service('D:\chromedriver-win32\chromedriver.exe'), options=chrome_options)
    return driver


def get_theater_links(driver, query):
    driver.get(query)
    time.sleep(1)
    number = 150

    addresslist = []

    for i in range(1, int(number) + 1):
        xpath = f"/html/body/table/tbody/tr[2]/td[3]/div/div/div[2]/div/table/tbody/tr[{i}]/td[1]/a"
        elem = driver.find_element(By.XPATH, xpath).get_attribute("href")
        addresslist.append(elem)

    return addresslist


def resize_image(image, max_height):
    # 이미지 크기를 최대 높이에 맞춰 조정
    width_percent = (max_height / float(image.size[1]))
    new_width = int((float(image.size[0]) * float(width_percent)))
    resized_image = image.resize((new_width, max_height), Image.LANCZOS)
    return resized_image

def image(driver, addresslist, save_directory, max_image_height=2000):
    scraped_image = []
    for index, lis in enumerate(addresslist):
        driver.get(lis)
        time.sleep(1)
        
        # 이미지 요소 찾기
        img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[5]/div/p/strong/img')
        title_element = driver.find_element(By.XPATH, '//*[@id="container"]/div[5]/div[1]/div[2]/div[1]/div/div[1]/h2')
        
        # 이미지가 없을 때 처리
        if not img_area_elements:
            img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[3]/div/p/strong/img')
            if not img_area_elements:
                img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[4]/div/p/strong/img')
            # 이미지 저장
            image_list = []
            for i, img_element in enumerate(img_area_elements):
                img_url = img_element.get_attribute("src")
                img_data = urllib.request.urlopen(img_url).read()

                # 파일 이름 생성
                filename=title_element.text.replace('/', '_').replace(':', '_')
                file_name= filename+"_part"+str(i + 1)+".png"
                file_path = os.path.join(save_directory, file_name)

                # 이미지 저장
                with open(file_path, "wb") as file:
                    file.write(img_data)
                    print(f"Saved {file_name}")

        else:
            # 이미지가 있는 경우 처리 코드
        
            # 이미지 저장
            image_list = []
            for i, img_element in enumerate(img_area_elements):
                img_url = img_element.get_attribute("src")
                img_data = urllib.request.urlopen(img_url).read()

                # 파일 이름 생성
                filename=title_element.text.replace('/', '_').replace(':', '_')
                file_name= filename+"_part"+str(i + 1)+".png"
                file_path = os.path.join(save_directory, file_name)

                # 이미지 저장
                with open(file_path, "wb") as file:
                    file.write(img_data)
                    print(f"Saved {file_name}")
        
        #7.이미지에서 문자추출
        for i, img_element in enumerate(img_area_elements):
            img_text=[]
            img_name='D:/캡스톤 디자인/연극포스터/'+filename+"_part"+str(i + 1)+".png"
            img1=np.array(Image.open(img_name))
            text=pytesseract.image_to_string(img1,config='--psm 6 --oem 1',lang='kor')
            img_text.append(text)
            #img_text.append('\n')
            #img_text.append(text)
            
            scraped_image.append({'Title': title_element.text, 'Text':text})#,'img_URL': img_url})
            #print(text)
            
    return scraped_image
        
def save_to_csv(data_list, save_path):
    if data_list:
        combined_data = pd.DataFrame(data_list)
        combined_data.to_csv(save_path, index=False)

In [21]:
current_directory = os.getcwd()
save_path = os.path.join(current_directory, "D:/연극_ocr_전처리전_제목0.csv")

driver = initialize_driver()
query = "http://ticket.interpark.com/TPGoodsList.asp?Ca=Dra&Sort=3"
addresslist = get_theater_links(driver, query)
scraped_image = image(driver, addresslist,'D:/캡스톤 디자인/연극포스터/')
save_to_csv(scraped_image, save_path)
driver.quit()

Saved 연극 너의 목소리가 들려_part1.png
Saved 연극 너의 목소리가 들려_part2.png
Saved 연극 너의 목소리가 들려_part3.png
Saved 대학로 1위 연극 〈쉬어매드니스〉_part1.png
Saved 대학로 1위 연극 〈쉬어매드니스〉_part2.png
Saved 행오버_part1.png
Saved 행오버_part2.png
Saved 행오버_part3.png
Saved 행오버_part4.png
Saved 연극 〈바닷마을 다이어리〉_part1.png
Saved 연극 〈바닷마을 다이어리〉_part2.png
Saved 연극 한뼘사이_part1.png
Saved 연극 한뼘사이_part2.png
Saved 연극 한뼘사이_part3.png
Saved 연극 한뼘사이_part4.png
Saved 연극 한뼘사이_part5.png
Saved 연극 한뼘사이_part6.png
Saved 연극 한뼘사이_part7.png
Saved 연극 한뼘사이_part8.png
Saved 연극 한뼘사이_part9.png
Saved 연극 한뼘사이_part10.png
Saved 연극 한뼘사이_part11.png
Saved 연극 한뼘사이_part12.png
Saved 연극 한뼘사이_part13.png
Saved 연극 한뼘사이_part14.png
Saved 연극 한뼘사이_part15.png
Saved 연극〈늘근도둑이야기〉_part1.png
Saved 연극〈늘근도둑이야기〉_part2.png
Saved 뮤직드라마 〈불편한 편의점〉_part1.png
Saved 뮤직드라마 〈불편한 편의점〉_part2.png
Saved 뮤직드라마 〈불편한 편의점〉_part3.png
Saved 뮤직드라마 〈불편한 편의점〉_part4.png
Saved 뮤직드라마 〈불편한 편의점〉_part5.png
Saved 코믹감동 휴먼판타지 연극 〈2호선 세입자〉_part1.png
Saved 코믹감동 휴먼판타지 연극 〈2호선 세입자〉_part2.png
Saved 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀_part

In [22]:
a=pd.read_csv("D:/연극_ocr_전처리전_제목0.csv")
a

,Title,Text
0,연극 너의 목소리가 들려,"0] 임이 79 90890 0000\n` 머크 <터의[목소리[""들려/ 대\n』 ..."
1,연극 너의 목소리가 들려,"@\n> `\n2\n1 >\n“> ㅜ\n, ..."
2,연극 너의 목소리가 들려,ㅇ\n1077\n2100 430 6130\n박선영 박선영 피설아 : 아 인 아 인 ...
3,대학로 1위 연극 〈쉬어매드니스〉,”-- -ㅋ- =\n콘텐츠박스 ...
4,대학로 1위 연극 〈쉬어매드니스〉,오이타 내더연어\n그 오소시 거으가\n.0010066 10월\n...
...,...,...
240,문화가 있는 날 Ⅳ NT Live［리어왕］- 대전,"트는 0"" _ 내\n ..."
241,뮤직드라마 〈망원동 브라더스〉,고 는 그노든(| 돋 |[.왔 [\n골때리는 그놈들이 돌.마왔다!\n ...
242,뮤직드라마 〈망원동 브라더스〉,1.00210700\n-오시는킬-\n서울특별시 마포구 어울마당로 94-8 61 제이...
243,뮤직드라마 〈망원동 브라더스〉,원작 김호연 작가 밀리언셀러 “불편한 편의점*이 공연으로 재탄생\n\n뮤직드라마 ...


In [23]:
result_df = a.groupby("Title")["Text"].apply(lambda x: "\n".join(x)).reset_index()
result_df

,Title,Text
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,| = [그 ...
1,(리얼타임 코믹연극) 택시안에서 - 부산,"、 、 웃음, 감동, 사랑이 시작되는 리얼타엄 코믹연극\n2019.03.15680)..."
2,(리얼타임 코믹연극) 택시안에서 - 서울,ㆍ ㆍ ㆍ 0” 로 변\...
3,(코믹연극) 달동네-부산,0 ~ 20006 웅새떼스 026 토\n그 206 6: 버 ~? ...
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,가을 바람 살학살항\n뼈 |\n로맨틱\n|\n205\n10넌 연숙 ...
...,...,...
141,［연극 on stage］ 띨뿌리 - 화성,"크 의닝 아Ｌ 1\n\n트리거워닝안내 ” ㅠㅠ ~,\n\n연극 <떨뿌리..."
142,［연애하기 좋은 날 : 당근거래］ - 전주,"~ 오 100 ^ 1\n. -헤\n개 |"" 0 ..."
143,［장애인 선예매］ 러브 앤 인포메이션,...
144,［창작공감: 연출］당신에게 닿는 길,9 홍익대대학로 작연출:\...


In [24]:
def initialize_driver():
    chrome_options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=Service('D:\chromedriver-win32\chromedriver.exe'), options=chrome_options)
    return driver

def get_theater_links(driver, query):
    driver.get(query)
    time.sleep(1)
    number = 160

    addresslist = []

    for i in range(1, int(number) + 1):
        xpath = f"/html/body/table/tbody/tr[2]/td[3]/div/div/div[2]/div/table/tbody/tr[{i}]/td[1]/a"
        elem = driver.find_element(By.XPATH, xpath).get_attribute("href")
        addresslist.append(elem)

    return addresslist


def resize_image(image, max_height):
    # 이미지 크기를 최대 높이에 맞춰 조정
    width_percent = (max_height / float(image.size[1]))
    new_width = int((float(image.size[0]) * float(width_percent)))
    resized_image = image.resize((new_width, max_height), Image.LANCZOS)
    return resized_image

def image(driver, addresslist, save_directory, max_image_height=2000):
    scraped_image = []
    for index, lis in enumerate(addresslist):
        driver.get(lis)
        time.sleep(1)
        
        # 이미지 요소 찾기
        img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[5]/div/p/strong/img')
        title_element = driver.find_element(By.XPATH, '//*[@id="container"]/div[5]/div[1]/div[2]/div[1]/div/div[1]/h2')
        
        # 이미지가 없을 때 처리
        if not img_area_elements:
            img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[3]/div/p/strong/img')
            if not img_area_elements:
                img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[4]/div/p/strong/img')
            # 이미지 저장
            image_list = []
            for i, img_element in enumerate(img_area_elements):
                img_url = img_element.get_attribute("src")
                img_data = urllib.request.urlopen(img_url).read()

                # 파일 이름 생성
                filename=title_element.text.replace('/', '_').replace(':', '_')
                file_name= filename+"_part"+str(i + 1)+".png"
                file_path = os.path.join(save_directory, file_name)

                # 이미지 저장
                with open(file_path, "wb") as file:
                    file.write(img_data)
                    print(f"Saved {file_name}")

        else:
            # 이미지가 있는 경우 처리 코드
        
            # 이미지 저장
            image_list = []
            for i, img_element in enumerate(img_area_elements):
                img_url = img_element.get_attribute("src")
                img_data = urllib.request.urlopen(img_url).read()

                # 파일 이름 생성
                filename=title_element.text.replace('/', '_').replace(':', '_')
                file_name= filename+"_part"+str(i + 1)+".png"
                file_path = os.path.join(save_directory, file_name)

                # 이미지 저장
                with open(file_path, "wb") as file:
                    file.write(img_data)
                    print(f"Saved {file_name}")
        
        #7.이미지에서 문자추출
        for i, img_element in enumerate(img_area_elements):
            img_text=[]
            img_name='D:/캡스톤 디자인/연극포스터/'+filename+"_part"+str(i + 1)+".png"
            img1=np.array(Image.open(img_name))
            text=pytesseract.image_to_string(img1,config='--psm 6 --oem 1',lang='kor')
            img_text.append(text)
            #img_text.append('\n')
            #img_text.append(text)
            
            scraped_image.append({'Title': title_element.text, 'Text':text})#,'img_URL': img_url})
            #print(text)
            
    return scraped_image
        
def save_to_csv(data_list, save_path):
    if data_list:
        combined_data = pd.DataFrame(data_list)
        combined_data.to_csv(save_path, index=False)

In [25]:
current_directory = os.getcwd()
save_path = os.path.join(current_directory, "D:/연극_ocr_전처리전_제목160.csv")

driver = initialize_driver()
query = "http://ticket.interpark.com/TPGoodsList.asp?Ca=Dra&Sort=3"
addresslist = get_theater_links(driver, query)
scraped_image = image(driver, addresslist,'D:/캡스톤 디자인/연극포스터/')
save_to_csv(scraped_image, save_path)
driver.quit()

Saved 연극 너의 목소리가 들려_part1.png
Saved 연극 너의 목소리가 들려_part2.png
Saved 연극 너의 목소리가 들려_part3.png
Saved 대학로 1위 연극 〈쉬어매드니스〉_part1.png
Saved 대학로 1위 연극 〈쉬어매드니스〉_part2.png
Saved 행오버_part1.png
Saved 행오버_part2.png
Saved 행오버_part3.png
Saved 행오버_part4.png
Saved 연극 〈바닷마을 다이어리〉_part1.png
Saved 연극 〈바닷마을 다이어리〉_part2.png
Saved 연극 한뼘사이_part1.png
Saved 연극 한뼘사이_part2.png
Saved 연극 한뼘사이_part3.png
Saved 연극 한뼘사이_part4.png
Saved 연극 한뼘사이_part5.png
Saved 연극 한뼘사이_part6.png
Saved 연극 한뼘사이_part7.png
Saved 연극 한뼘사이_part8.png
Saved 연극 한뼘사이_part9.png
Saved 연극 한뼘사이_part10.png
Saved 연극 한뼘사이_part11.png
Saved 연극 한뼘사이_part12.png
Saved 연극 한뼘사이_part13.png
Saved 연극 한뼘사이_part14.png
Saved 연극 한뼘사이_part15.png
Saved 연극〈늘근도둑이야기〉_part1.png
Saved 연극〈늘근도둑이야기〉_part2.png
Saved 뮤직드라마 〈불편한 편의점〉_part1.png
Saved 뮤직드라마 〈불편한 편의점〉_part2.png
Saved 뮤직드라마 〈불편한 편의점〉_part3.png
Saved 뮤직드라마 〈불편한 편의점〉_part4.png
Saved 뮤직드라마 〈불편한 편의점〉_part5.png
Saved 코믹감동 휴먼판타지 연극 〈2호선 세입자〉_part1.png
Saved 코믹감동 휴먼판타지 연극 〈2호선 세입자〉_part2.png
Saved 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀_part

In [26]:
a=pd.read_csv("D:/연극_ocr_전처리전_제목160.csv")
a

,Title,Text
0,연극 너의 목소리가 들려,"0] 임이 79 90890 0000\n` 머크 <터의[목소리[""들려/ 대\n』 ..."
1,연극 너의 목소리가 들려,"@\n> `\n2\n1 >\n“> ㅜ\n, ..."
2,연극 너의 목소리가 들려,ㅇ\n1077\n2100 430 6130\n박선영 박선영 피설아 : 아 인 아 인 ...
3,대학로 1위 연극 〈쉬어매드니스〉,”-- -ㅋ- =\n콘텐츠박스 ...
4,대학로 1위 연극 〈쉬어매드니스〉,오이타 내더연어\n그 오소시 거으가\n.0010066 10월\n...
...,...,...
255,서교동에서 죽다 - 서울 꿈빛극장,"가 ~ ~ …~\n- “""% ㅋ\n주 ~ ..."
256,섬,@ ' ...
257,섹시코미디연극 나의PS파트너 (청주),때 다 이 [ 1\n내가 그 날 열마Ｌ\n~ 『\...
258,쇼팔다 미친 유령 극단,"8 마지막 주, 10잇 번페 주 공연 없음\n2023년 9월 1일부터 10월 15일..."


In [27]:
result_df = a.groupby("Title")["Text"].apply(lambda x: "\n".join(x)).reset_index()
result_df

,Title,Text
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,| = [그 ...
1,(리얼타임 코믹연극) 택시안에서 - 부산,"、 、 웃음, 감동, 사랑이 시작되는 리얼타엄 코믹연극\n2019.03.15680)..."
2,(리얼타임 코믹연극) 택시안에서 - 서울,ㆍ ㆍ ㆍ 0” 로 변\...
3,(코믹연극) 달동네-부산,0 ~ 20006 웅새떼스 026 토\n그 206 6: 버 ~? ...
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,가을 바람 살학살항\n뼈 |\n로맨틱\n|\n205\n10넌 연숙 ...
...,...,...
151,［연극 on stage］ 띨뿌리 - 화성,"크 의닝 아Ｌ 1\n\n트리거워닝안내 ” ㅠㅠ ~,\n\n연극 <떨뿌리..."
152,［연애하기 좋은 날 : 당근거래］ - 전주,"~ 오 100 ^ 1\n. -헤\n개 |"" 0 ..."
153,［장애인 선예매］ 러브 앤 인포메이션,...
154,［창작공감: 연출］당신에게 닿는 길,9 홍익대대학로 작연출:\...
